<a href="https://colab.research.google.com/github/doomguy0991/dls_c/blob/main/C4%20-%20Convolutional%20Neural%20Networks/Notes/Readme.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import os

# Check if running in Google Colab
try:
    import google.colab
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

if IN_COLAB:
    print("Setting up Colab environment...")

    # 1. Clone the repository
    repo_url = "https://github.com/doomguy0991/dls_c.git"
    # Only clone if the directory doesn't exist yet
    if not os.path.exists("/content/dls_c"):
        !git clone $repo_url /content/dls_c

    # 2. Change working directory to the notebook's location so relative paths for libraries work imports
    %cd "/content/dls_c/C4 - Convolutional Neural Networks/Notes"

    print("Setup complete. You can now run the rest of the notebook.")
else:
    print("Running locally or out of Colab. No setup needed.")


# Convolutional Neural Networks

This is the fourth course of the deep learning specialization at [Coursera](https://www.coursera.org/specializations/deep-learning) which is moderated by [DeepLearning.ai](http://deeplearning.ai/). The course is taught by Andrew Ng.

## Table of contents

* [Convolutional Neural Networks](#convolutional-neural-networks)
   * [Table of contents](#table-of-contents)
   * [Course summary](#course-summary)
   * [Foundations of CNNs](#foundations-of-cnns)
      * [Computer vision](#computer-vision)
      * [Edge detection example](#edge-detection-example)
      * [Padding](#padding)
      * [Strided convolution](#strided-convolution)
      * [Convolutions over volumes](#convolutions-over-volumes)
      * [One Layer of a Convolutional Network](#one-layer-of-a-convolutional-network)
      * [A simple convolution network example](#a-simple-convolution-network-example)
      * [Pooling layers](#pooling-layers)
      * [Convolutional neural network example](#convolutional-neural-network-example)
      * [Why convolutions?](#why-convolutions)
   * [Deep convolutional models: case studies](#deep-convolutional-models-case-studies)
      * [Why look at case studies?](#why-look-at-case-studies)
      * [Classic networks](#classic-networks)
      * [Residual Networks (ResNets)](#residual-networks-resnets)
      * [Why ResNets work](#why-resnets-work)
      * [Network in Network and 1×1 convolutions](#network-in-network-and-1-X-1-convolutions)
      * [Inception network motivation](#inception-network-motivation)
      * [Inception network (GoogleNet)](#inception-network-googlenet)
      * [Using Open-Source Implementation](#using-open-source-implementation)
      * [Transfer Learning](#transfer-learning)
      * [Data Augmentation](#data-augmentation)
      * [State of Computer Vision](#state-of-computer-vision)
   * [Object detection](#object-detection)
      * [Object Localization](#object-localization)
      * [Landmark Detection](#landmark-detection)
      * [Object Detection](#object-detection-1)
      * [Convolutional Implementation of Sliding Windows](#convolutional-implementation-of-sliding-windows)
      * [Bounding Box Predictions](#bounding-box-predictions)
      * [Intersection Over Union](#intersection-over-union)
      * [Non-max Suppression](#non-max-suppression)
      * [Anchor Boxes](#anchor-boxes)
      * [YOLO Algorithm](#yolo-algorithm)
      * [Region Proposals (R-CNN)](#region-proposals-r-cnn)
   * [Special applications: Face recognition &amp; Neural style transfer](#special-applications-face-recognition--neural-style-transfer)
      * [Face Recognition](#face-recognition)
         * [What is face recognition?](#what-is-face-recognition)
         * [One Shot Learning](#one-shot-learning)
         * [Siamese Network](#siamese-network)
         * [Triplet Loss](#triplet-loss)
         * [Face Verification and Binary Classification](#face-verification-and-binary-classification)
      * [Neural Style Transfer](#neural-style-transfer)
         * [What is neural style transfer?](#what-is-neural-style-transfer)
         * [What are deep ConvNets learning?](#what-are-deep-convnets-learning)
         * [Cost Function](#cost-function)
         * [Content Cost Function](#content-cost-function)
         * [Style Cost Function](#style-cost-function)
         * [1D and 3D Generalizations](#1d-and-3d-generalizations)
   * [Extras](#extras)
      * [Keras](#keras)

## Course summary

Here is the course summary as given on the course [link](https://www.coursera.org/learn/convolutional-neural-networks):

> This course will teach you how to build convolutional neural networks and apply it to image data. Thanks to deep learning, computer vision is working far better than just two years ago, and this is enabling numerous exciting applications ranging from safe autonomous driving, to accurate face recognition, to automatic reading of radiology images.
>
> You will:
> - Understand how to build a convolutional neural network, including recent variations such as residual networks.
> - Know how to apply convolutional networks to visual detection and recognition tasks.
> - Know to use neural style transfer to generate art.
> - Be able to apply these algorithms to a variety of image, video, and other 2D or 3D data.
>
> This is the fourth course of the Deep Learning Specialization.

## Foundation of Convolutional Neural Network

### Computer vision

Computer Vision (CV) is a rapidly advancing field of deep learning that enables machines to "see" and interpret visual data. It powers everything from self-driving cars (identifying pedestrians) and Face ID to **Neural Style Transfer** (applying an artist's style to a photo).

#### 1. The "Large Image" Challenge
In traditional fully connected networks, large images create a **parameter explosion**:
*   **64x64 pixel image:** Small, only ~12,288 input features.
*   **1000x1000 pixel image:** 1 million pixels $\times$ 3 color channels (RGB) = **3 million input features**.
*   If the first hidden layer has only 1,000 units, the weight matrix would have **3 billion parameters**. This is computationally expensive and leads to extreme **overfitting**.

**Solution:** Convolutional Neural Networks (CNNs) use the **Convolution** operation to reduce parameters and detect patterns efficiently.

<br>



### The Convolution Operation: Edge Detection



Convolution is the fundamental building block of CNNs. It allows the network to identify low-level features like edges before building up to complex objects.

#### 1. Mechanics of Convolution
To perform a convolution, you slide a small matrix called a **Filter** (or **Kernel**) over an input image and calculate the sum of element-wise products.

*   **Input Image:** Size $n \times n$
*   **Filter:** Size $f \times f$
*   **Output (Feature Map):** Size $(n - f + 1) \times (n - f + 1)$

**Example:** A $6 \times 6$ image convolved with a $3 \times 3$ filter results in a $4 \times 4$ output.



<br>

#### 2. Vertical vs. Horizontal Edge Detection
Filters are designed to react to specific changes in pixel intensity (brightness).

*   **Vertical Edge Filter:** Detects transitions from light to dark (or vice versa) horizontally.
    \begin{bmatrix} 1 & 0 & -1 \\ 1 & 0 & -1 \\ 1 & 0 & -1 \end{bmatrix}
*   **Horizontal Edge Filter:** Detects transitions vertically.
     \begin{bmatrix} 1 & 1 & 1 \\ 0 & 0 & 0 \\ -1 & -1 & -1 \end{bmatrix}
  - An example of convolution operation to detect vertical edges:
    - ![](https://github.com/doomguy0991/dls_c/blob/main/C4%20-%20Convolutional%20Neural%20Networks/Notes/Images/01.png?raw=1)

#### 3. Specialized Filters
Researchers have historically designed specific filters to improve accuracy:
*   **Sobel Filter:** Weights the central row/column more heavily to increase robustness to noise.
*   **Scharr Filter:** An even more aggressive weighting for detecting very sharp edges.

### Padding in CNNs



In deep neural networks, applying multiple convolutions causes the image to shrink significantly and results in the loss of information at the edges. **Padding** is the technique of adding extra borders of pixels (usually zeros) around the input image to solve these issues.



<br>

#### 1. Why do we need Padding?
Without padding, two major problems occur:
1.  **Shrinkage:** If you have a 100-layer network and each layer shrinks the image by a few pixels, you will eventually run out of pixels before reaching the end of the network.
2.  **Edge Information Loss:** Pixels in the center of an image are overlapped by the filter many times, but corner/edge pixels are only "touched" once. Padding ensures edge pixels contribute more to the output.

<br>

#### 2. The Math of Padding
If an $n \times n$ image is convolved with an $f \times f$ filter and padding $p$, the output dimension is:
$$(n + 2p - f + 1) \times (n + 2p - f + 1)$$

*   **$n$**: Input size
*   **$p$**: Padding amount (number of pixels added to each side)
*   **$f$**: Filter size

<br>

#### 3. Types of Convolution
There are two standard choices for padding in deep learning:

##### A. Valid Convolution
*   **Definition:** No padding ($p = 0$).
*   **Effect:** The image shrinks every time.
*   **Output Size:** $(n - f + 1) \times (n - f + 1)$

##### B. Same Convolution
*   **Definition:** Padding is added so that the **output size is exactly the same as the input size**.
*   **Formula to find $p$:** To make the output $n$, we solve the math to get: $$p = \frac{f - 1}{2}$$

<br>

#### 4. Why are Filters ($f$) usually Odd?
You will rarely see $2 \times 2$ or $4 \times 4$ filters in research. By convention, $f$ is almost always odd (e.g., $3 \times 3$, $5 \times 5$) for two reasons:
1.  **Symmetric Padding:** If $f$ is odd, the "Same" convolution padding $p$ is an integer. If $f$ were even, you would need asymmetric padding (e.g., more on the left than the right).
2.  **Central Pixel:** Odd-sized filters have a specific **center pixel**, which makes it easier to track the position and orientation of the filter relative to the image.

#### For Computer vision
In your future Computer Vision work, you will mostly use **"Same"** convolutions. This allows you to build very deep architectures (like ResNet or VGG) without worrying about your spatial dimensions disappearing before the final layer.

Do you see how the choice of an odd filter size like $3 \times 3$ makes the padding math much cleaner?

### Strided Convolutions



Strided convolution is a variation of the convolution operation where the filter "jumps" over a specified number of pixels instead of sliding one by one. This is primarily used to reduce the spatial dimensions (width and height) of the image.

<br>

#### 1. The Stride ($s$)
The **Stride** ($s$) is the step size.
*   **Stride = 1:** The filter moves one pixel at a time (standard).
*   **Stride = 2:** The filter skips one pixel, jumping to the second position.

*   **Why use Stride > 1?** It's an alternative to "Pooling" layers. It allows the network to reduce the size of the feature maps, which saves memory and computation while increasing the "Receptive Field" (allowing a single pixel in the next layer to see a larger portion of the original image).
*   **Usage Tip:** Most modern architectures use **Stride = 1** for standard layers and **Stride = 2** specifically when they want to reduce the image dimensions.

#### 2. Output Dimension Formula
If an $n \times n$ image is convolved with an $f \times f$ filter using padding $p$ and stride $s$, the output size is:

$$\text{Output Size} = \left\lfloor \frac{n + 2p - f}{s} + 1 \right\rfloor$$

*   **The Floor Operation ($\lfloor \rfloor$):** We round down. If the filter "hangs over" the edge of the image at its final step, we simply **ignore** that step. The filter must be fully contained within the image (or padded image) boundaries to produce an output.

> **Mathematical Simulation:**
> Let $n=7$ (input), $f=3$ (filter), $p=0$ (no padding), and $s=2$ (stride).
> *   Calculation: $(7 + 0 - 3) / 2 + 1 = \mathbf{3}$
> *   The output is a $3 \times 3$ matrix.

<br>

#### 3. Cross-Correlation vs. Convolution
There is a technical difference between how mathematicians and deep learning researchers define "convolution":

*   **Math/Signal Processing:** You must **flip** the filter horizontally and vertically before sliding it. This provides a property called *associativity*.
*   **Deep Learning:** We skip the flip and just perform the element-wise product. Technically, this is called **Cross-correlation**, but we call it "Convolution" by convention.
*   **Why skip the flip?** In deep learning, the filter values are **learned weights**. If the network needs a flipped filter, it will just learn the flipped values directly. Skipping the flip simplifies the code and works just as well.

<br>


### Convolutions Over Volumes



When working with color images (RGB) or layers within a deep network, convolutions happen in three dimensions ($Height \times Width \times Channels$).

<br>

#### 1. The Channel Rule
The most important rule in 3D convolutions is that the **number of channels in the filter must match the number of channels in the input**.

*   **Input Image:** $6 \times 6 \times \mathbf{3}$ (RGB)
*   **Filter:** $3 \times 3 \times \mathbf{3}$
*   **Channels (Depth):** This 3 represents the Red, Green, and Blue layers.

<br>

#### 2. How the Math Works
Instead of 9 multiplications (for $3 \times 3$), the computer now performs **27 multiplications** ($3 \times 3 \times 3$) at each position and sums them into a **single number**.
*   The filter slides through the height and width, but it covers the entire depth of the input at once.
*   **Crucial Point:** Even though the input and filter are 3D, the output of **one** filter is a **2D matrix** (e.g., $4 \times 4 \times 1$).

<br>

#### 3. Multiple Filters (Creating Volume)
In a real CNN, we don't just want to detect one feature (like vertical edges). We want to detect many (horizontal edges, colors, textures, etc.).

*   If you use **10 different filters**, you get **10 different $4 \times 4$ output maps**.
*   We stack these maps together to create a new 3D volume: $4 \times 4 \times 10$.



#### 4. Dimension Summary
If your input is $n \times n \times n_c$ and you convolve it with $n_f$ filters of size $f \times f \times n_c$:

$$\text{Output Shape} = (n - f + 1) \times (n - f + 1) \times n_f$$

*   **$n_c$**: Number of input channels (e.g., 3 for RGB).
*   **$n_f$**: Number of filters (this becomes the "channels" for the next layer).

<br>

### Anatomy of a Single CNN Layer


A single layer in a Convolutional Neural Network (CNN) performs a transformation similar to a standard fully connected layer ($z = wa + b$), but it uses the **convolution** operation to maintain spatial structure and drastically reduce the number of parameters.

<br>

#### 1. The Forward Propagation Steps
To go from the input activation $a^{[l-1]}$ to the next layer's activation $a^{[l]}$, the network performs three specific steps:

1.  **Convolution:** The input volume is convolved with $n_c^{[l]}$ filters. Filter == W[L], Input == a[L-1].

2.  **Add Bias:** A single real number (bias $b$) is added to each element of the filter's output (using Python broadcasting).
3.  **Activation:** A non-linear function (typically **ReLU**) is applied to the result.

**The Logic:**
$$a^{[l]} = g(\text{Conv}(a^{[l-1]}, W^{[l]}) + b^{[l]})$$



<br>

#### 2. Layer Notation Summary
If layer $l$ is a convolutional layer, the following notations apply:

| Property | Symbol / Formula |
| :--- | :--- |
| **Filter Size** | $f^{[l]}$ |
| **Padding / Stride** | $p^{[l]}$ / $s^{[l]}$ |
| **Input Dimensions ( $a^{[0]}$ )** | $n_H^{[l-1]} \times n_W^{[l-1]} \times n_c^{[l-1]}  $|
| **Output Dimensions (activation $a^{[l]}$ )** | $n_H^{[l]} \times n_W^{[l]} \times n_c^{[l]}$ |
| **Output Dimensionsfor m examples (activation $A^{[l]}$ )** | $n_H^{[l]} \times n_W^{[l]} \times n_c^{[l]}$ |
| **Output Height/Width** | $n_{H/W}^{[l]} = \lfloor \frac{n_{H/W}^{[l-1]} + 2p^{[l]} - f^{[l]}}{s^{[l]}} + 1 \rfloor$ |
| **Filter Dimensions** | $f^{[l]} \times f^{[l]} \times n_c^{[l-1]}$ |
| **#No of filters** | $n_c^{[l]}$ |
| **Weight Tensor ($W^{[l]}$)** | $f^{[l]} \times f^{[l]} \times n_c^{[l-1]} \times n_c^{[l]}$ |
| **Bias ($b^{[l]}$)** | $1 \times 1 \times 1 \times n_c^{[l]}$ |

<br>

### A simple convolution network example


This example demonstrates how spatial dimensions are reduced using different strides and filter sizes.

* **Input Image ($a^{[0]}$):** $39 \times 39 \times 3$
    * $n_H = 39, n_W = 39, n_c = 3$ (RGB)

* **Layer 1 (Convolutional):**
    * **Parameters:** $f=3, s=1, p=0$, with **10 filters**.
    * **Output ($a^{[1]}$):** $37 \times 37 \times 10$
    * *Math:* $\frac{39 + 0 - 3}{1} + 1 = 37$

* **Layer 2 (Convolutional):**
    * **Parameters:** $f=5, s=2, p=0$, with **20 filters**.
    * **Output ($a^{[2]}$):** $17 \times 17 \times 20$
    * *Math:* $\lfloor \frac{37 + 0 - 5}{2} + 1 \rfloor = \lfloor 16 + 1 \rfloor = 17$
    * **Insight:** Shrinking happens much faster here because the **stride ($s=2$)** effectively halves the output size.

* **Layer 3 (Convolutional):**
    * **Parameters:** $f=5, s=2, p=0$, with **40 filters**.
    * **Output ($a^{[3]}$):** $7 \times 7 \times 40$
    * *Math:* $\lfloor \frac{17 + 0 - 5}{2} + 1 \rfloor = \lfloor 6 + 1 \rfloor = 7$

* **Layer 4 (Fully Connected + Softmax):**
    * The 3D volume from Layer 3 ($7 \times 7 \times 40$) is **flattened** into a 1D vector.
    * **Input to FC:** $7 \times 7 \times 40 = \mathbf{1,960}$ **units**.
    * This vector is then fed into a Softmax layer for multi-class classification.

#### The Three Pillars: Layer Types
Most CNN architectures are composed of these three fundamental layer types:

| Layer Type | Abbreviation | Purpose |
| :--- | :--- | :--- |
| **Convolutional** | **#Conv** | Feature extraction through learned filters. |
| **Pooling** | **#Pool** | Downsampling to reduce spatial size and parameter count (e.g., Max Pool). |
| **Fully Connected** | **#FC** | Standard neural network layer used at the end for classification/regression. |



### Pooling Layers



Pooling layers are used in ConvNets primarily to **reduce the spatial size** (downsampling) of the representation. This speeds up computation, reduces the number of parameters in subsequent layers (preventing overfitting), and makes feature detection more robust to small distortions or shifts in the image.

<br>

#### Max Pooling
This is the most common type of pooling. For every region covered by the filter, it selects the **maximum value**.

* **Logic:** If we treat the numbers as "feature activations," a high number means a feature (like a cat's ear or a vertical edge) was detected. Max pooling says: "If this feature exists *anywhere* in this small region, preserve it."
* **Hyperparameters:** Usually $f=2, s=2$. This effectively halves the height and width of the image.



<br>

#### Average Pooling
Instead of taking the maximum, this layer calculates the **average value** of all pixels in the filter region.

* **Usage:** It is much less common than Max Pooling in early/middle layers.
* **Exception:** Often used at the very end of deep architectures (Global Average Pooling) to collapse a volume like $7 \times 7 \times 1000$ into a $1 \times 1 \times 1000$ vector before the final classification.

<br>

#### Crucial Understandings

* **No Learnable Parameters:** Unlike Convolutional layers, pooling has **nothing for Gradient Descent to learn**. Once you choose the filter size ($f$) and stride ($s$), the operation is a fixed mathematical function.
* **Channel Independence:** Pooling is performed on each channel **independently**. If the input is $n_H \times n_W \times n_C$, the output will still have $n_C$ channels. The depth never changes during a pooling operation.
* **Padding:** It is very rare to use padding in pooling layers. By default, $p = 0$.

<br>

#### Dimensions & Formula
The output size follows the same formula as the convolution layer:

$$n_{H/W}^{[l]} = \left\lfloor \frac{n_{H/W}^{[l-1]} + 2p - f}{s} + 1 \right\rfloor$$

**Common Hyperparameter Combinations:**
* $f=2, s=2$ (Most common: reduces size by 50%)
* $f=3, s=2$ (Overlapping pooling)

### Why Convolutions?



Compared to standard Fully Connected (FC) layers, Convolutional layers are far more efficient for image data. This efficiency allows us to build deep networks for high-resolution images without the "parameter explosion" that would otherwise occur.

<br>

#### 1. Two Main Advantages

##### A. Parameter Sharing
A feature detector (like a vertical edge filter) that is useful in the top-left of an image is likely useful in the bottom-right as well.
* **How it works:** Instead of learning different weights for every pixel location, we learn **one filter** and slide it across the entire image.
* **Result:** All positions in the input image share the same parameters. This drastically reduces the number of values the model needs to learn.



##### B. Sparsity of Connections
In a convolutional layer, each output value depends only on a small number of inputs (the size of the filter), not the entire image.
* **Example:** In a $3 \times 3$ convolution, a single output unit is "connected" to only 9 input pixels.
* **Result:** This localized connection makes the network **Translation Invariant**. Because the model looks at local patterns, a "cat" shifted by 10 pixels is still recognized as a cat because the same filter will eventually slide over it and "fire."



<br>

#### 2. Parameter Comparison: FC vs. Conv
Consider a small $32 \times 32 \times 3$ image (3,072 features) mapped to a $28 \times 28 \times 6$ output (4,704 features).

* **Fully Connected Layer:** Every input connects to every output.
    * Parameters $\approx 3,072 \times 4,704 \approx \mathbf{14.4\text{ million}}$
* **Convolutional Layer ($5 \times 5$ filter, 6 filters):**
    * Parameters $= 6 \times (5 \times 5 \times 3 + 1) = \mathbf{456}$

The reduction from 14 million to 456 parameters is staggering. This efficiency prevents **overfitting** and makes training on smaller datasets feasible.

<br>

#### 3. Putting It All Together: The Training Flow
To build an end-to-end system (like a cat detector), we stack these blocks together:

1.  **Architecture:** Input $\rightarrow$ [Conv $\rightarrow$ Pool] layers $\rightarrow$ [FC] layers $\rightarrow$ **Softmax Output**.
2.  **Cost Function ($J$):** We define the cost as the average loss ($L$) across all $m$ training examples:

    $$J = \frac{1}{m} \sum_{i=1}^{m} L(\hat{y}^{(i)}, y^{(i)})$$
3.  **Optimization:** We use algorithms like **Adam**, **Momentum**, or **RMSProp** to update all weights ($W$) and biases ($b$) to minimize $J$.

<br>



## Deep convolutional models: case studies

### Why look at case studies?

- We learned about Conv layer, pooling layer, and fully connected layers. It turns out that computer vision researchers spent the past few years on how to put these layers together.
- To get some intuitions you have to see the examples that has been made.
- Some neural networks architecture that works well in some tasks can also work well in other tasks.
- Here are some classical CNN networks:
  - **LeNet-5**
  - **AlexNet**
  - **VGG**
- The ***best CNN architecture that won the last ImageNet competition*** is called **ResNet** and it has ***152 layers***!
- There are also an architecture called **Inception** that was made by Google that are very useful to learn and apply to your tasks.
- Reading and trying the mentioned models can boost you and give you a lot of ideas to solve your task.

### Classic networks



- In this section we will talk about classic networks which are **LeNet-5**, **AlexNet**, and **VGG**.

- **LeNet-5**

  - The goal for this model was to identify handwritten digits in a `32x32x1` gray image. Here are the drawing of it:
  - ![](https://github.com/doomguy0991/dls_c/blob/main/C4%20-%20Convolutional%20Neural%20Networks/Notes/Images/05.png?raw=1)
  - This model was published in 1998. The last layer wasn't using softmax back then.
  - It has 60k parameters.
  - The dimensions of the image decreases as the number of channels increases.
  - `Conv ==> Pool ==> Conv ==> Pool ==> FC ==> FC ==> softmax` this type of arrangement is quite common.
  - The activation function used in the paper was Sigmoid and Tanh. Modern implementation uses RELU in most of the cases.
  - [[LeCun et al., 1998. Gradient-based learning applied to document recognition]](http://ieeexplore.ieee.org/document/726791/?reload=true)

- **AlexNet**

  - Named after Alex Krizhevsky who was the first author of this paper. The other authors includes Geoffrey Hinton.

  - The goal for the model was the ImageNet challenge which classifies images into 1000 classes. Here are the drawing of the model:

  - ![](https://github.com/doomguy0991/dls_c/blob/main/C4%20-%20Convolutional%20Neural%20Networks/Notes/Images/06.png?raw=1)

  - Summary:

    - ```
      Conv => Max-pool => Conv => Max-pool => Conv => Conv => Conv => Max-pool ==> Flatten ==> FC ==> FC ==> Softmax
      ```

  - Similar to LeNet-5 but bigger.

  - Has 60 Million parameter compared to 60k parameter of LeNet-5.

  - It used the RELU activation function.

  - The original paper contains Multiple GPUs and Local Response normalization (RN).

    - Multiple GPUs were used because the GPUs were not so fast back then.
    - Researchers proved that Local Response normalization doesn't help much so for now don't bother yourself for understanding or implementing it.

  - This paper convinced the computer vision researchers that deep learning is so important.

  - [[Krizhevsky et al., 2012. ImageNet classification with deep convolutional neural networks]](https://papers.nips.cc/paper/4824-imagenet-classification-with-deep-convolutional-neural-networks.pdf)

- **VGG-16**

  - A modification for AlexNet.
  - Instead of having a lot of hyperparameters lets have some simpler network.
  - Focus on having only these blocks:
    - CONV = 3 X 3 filter, s = 1, same  
    - MAX-POOL = 2 X 2 , s = 2
  - Here are the architecture:
    - ![](https://github.com/doomguy0991/dls_c/blob/main/C4%20-%20Convolutional%20Neural%20Networks/Notes/Images/07.png?raw=1)
  - This network is large even by modern standards. It has around 138 million parameters.
    - Most of the parameters are in the fully connected layers.
  - It has a total memory of 96MB per image for only forward propagation!
    - Most memory are in the earlier layers.
  - Number of filters increases from 64 to 128 to 256 to 512. 512 was made twice.
  - Pooling was the only one who is responsible for shrinking the dimensions.
  - There are another version called **VGG-19** which is a bigger version. But most people uses the VGG-16 instead of the VGG-19 because it does the same.
  - VGG paper is attractive it tries to make some rules regarding using CNNs.
  - [[Simonyan & Zisserman 2015. Very deep convolutional networks for large-scale image recognition]](https://arxiv.org/abs/1409.1556)


### Residual Networks (ResNets)



- Very, very deep NNs are difficult to train because of vanishing and exploding gradients problems.
- In this section we will learn about skip connection which makes you take the activation from one layer and suddenly feed it to another layer even much deeper in NN which allows you to train large NNs even with layers greater than 100.
- **Residual block**
  - ResNets are built out of some Residual blocks.
  - ![](https://github.com/doomguy0991/dls_c/blob/main/C4%20-%20Convolutional%20Neural%20Networks/Notes/Images/08.png?raw=1)
  - They add a shortcut/skip connection before the second activation.
  - The authors of this block find that you can train a deeper NNs using stacking this block.
  - [[He et al., 2015. Deep residual networks for image recognition]](https://arxiv.org/abs/1512.03385)
- **Residual Network**
  - Are a NN that consists of some Residual blocks.
  - ![](https://github.com/doomguy0991/dls_c/blob/main/C4%20-%20Convolutional%20Neural%20Networks/Notes/Images/09.png?raw=1)
  - These networks can go deeper without hurting the performance. In the normal NN - Plain networks - the theory tell us that if we go deeper we will get a better solution to our problem, but because of the vanishing and exploding gradients problems the performance of the network suffers as it goes deeper. Thanks to Residual Network we can go deeper as we want now.
  - ![](https://github.com/doomguy0991/dls_c/blob/main/C4%20-%20Convolutional%20Neural%20Networks/Notes/Images/10.png?raw=1)
  - On the left is the normal NN and on the right are the ResNet. As you can see the performance of ResNet increases as the network goes deeper.
  - In some cases going deeper won't effect the performance and that depends on the problem on your hand.
  - Some people are trying to train 1000 layer now which isn't used in practice.
  - [He et al., 2015. Deep residual networks for image recognition]

#### Why ResNets work

- Lets see some example that illustrates why resNet work.

  - We have a big NN as the following:

    - `X --> Big NN --> a[l]`

  - Lets add two layers to this network as a residual block:

    - `X --> Big NN --> a[l] --> Layer1 --> Layer2 --> a[l+2]`
    - And a`[l]` has a direct connection to `a[l+2]`

  - Suppose we are using RELU activations.

  - Then:

    - ```
      a[l+2] = g( z[l+2] + a[l] )
      	   = g( W[l+2] a[l+1] + b[l+2] + a[l] )
      ```

  - Let's look at the math. If we use L2 regularization (weight decay), the weights $W^{[l+2]}$ and biases $b^{[l+2]}$ will tend toward zero.
  - The formula becomes: $a^{[l+2]} = g(0 + a^{[l]})$Since we use ReLU (which leaves positive numbers unchanged), $g(a^{[l]}) = a^{[l]}$.
  <br>

- **Conclusion:**
Even if the new layers learn absolutely nothing, the skip connection simply passes the previous information forward ($a^{[l+2]} = a^{[l]}$). Therefore, adding extra layers cannot hurt the performance of the network. If the new layers learn something useful, the performance improves; if not, they act as an identity function.
<br>

- **Handling Dimension Mismatches:**
  - For the addition $z^{[l+2]} + a^{[l]}$ to work, both vectors must have the exact same dimensions.
  - ResNets heavily rely on "Same" Convolutions so the spatial size doesn't shrink during the block.
  - When dimensions do change (e.g., via a pooling layer or strided convolution), we multiply $a^{[l]}$ by a matrix $W_s$ to match the new dimensions:$$a^{[l+2]} = g(z^{[l+2]} + W_s a^{[l]})$$(Note: $W_s$ can be learned parameters or simply a fixed zero-padding operation).

- Using a skip-connection helps the gradient to backpropagate and thus helps you to train deeper networks

- Lets take a look at ResNet on images.

  - Here are the architecture of **ResNet-34**:
  - ![](https://github.com/doomguy0991/dls_c/blob/main/C4%20-%20Convolutional%20Neural%20Networks/Notes/Images/resNet.jpg?raw=1)
  - All the 3x3 Conv are same Convs.
  - Keep it simple in design of the network.
  - spatial size /2 => # filters x2
  - No FC layers, No dropout is used.
  - Two main types of blocks are used in a ResNet, depending mainly on whether the input/output dimensions are same or different. You are going to implement both of them.
  - The dotted lines is the case when the dimensions are different. To solve then they down-sample the input by 2 and then pad zeros to match the two dimensions. There's another trick which is called bottleneck which we will explore later.

- Useful concept (**Spectrum of Depth**):

  - ![](https://github.com/doomguy0991/dls_c/blob/main/C4%20-%20Convolutional%20Neural%20Networks/Notes/Images/12.png?raw=1)
  - Taken from [icml.cc/2016/tutorials/icml2016_tutorial_deep_residual_networks_kaiminghe.pdf](icml.cc/2016/tutorials/icml2016_tutorial_deep_residual_networks_kaiminghe.pdf)

- Residual blocks types:

  - Identity block:
    - ![](https://github.com/doomguy0991/dls_c/blob/main/C4%20-%20Convolutional%20Neural%20Networks/Notes/Images/16.png?raw=1)
    - Hint the conv is followed by a batch norm `BN` before `RELU`. Dimensions here are same.
    - This skip is over 2 layers. The skip connection can jump n connections where n>2
    - This drawing represents [Keras](https://keras.io/) layers.
  - The convolutional block:
    - ![](https://github.com/doomguy0991/dls_c/blob/main/C4%20-%20Convolutional%20Neural%20Networks/Notes/Images/17.png?raw=1)
    - The conv can be bottleneck 1 x 1 conv

### Network in Network (NiN) & 1x1 Convolutions



A **1x1 Convolution** (also known as **Network in Network**) is a powerful tool in Convolutional Neural Networks (CNNs). While it sounds like simple multiplication, it performs a much more complex role when applied to multi-layered images.

<br>

#### 1. What does it actually do?

If you apply a $1 \times 1$ filter to a 2D image (1 channel), it just multiplies every pixel by a single number. However, modern CNNs use volumes with many channels (e.g., $6 \times 6 \times 32$).

* **The Slice Effect:** At each of the 36 positions (in a $6 \times 6$ grid), the $1 \times 1$ convolution looks at all 32 numbers in that specific "slice."
* **Mini-Neural Network:** It treats those 32 numbers as inputs to a single neuron, multiplies them by weights, adds them up, and applies a **ReLU** activation.
* **Output:** If you use 5 different $1 \times 1$ filters, your output becomes $6 \times 6 \times 5$.

<br>

#### 2. Why is this useful?

There are two primary reasons why researchers use $1 \times 1$ convolutions in architectures like **Inception** and **ResNet**:

- ##### ***A. Dimensionality Reduction (Shrinking Channels)***

  Pooling layers (like Max Pooling) are great for shrinking height and width, but they cannot shrink the number of channels.

  * **Example:** You have a volume of $28 \times 28 \times 192$. You want to reduce the depth to 32.
  * **Solution:** Apply 32 filters of size $1 \times 1 \times 192$. The result is a $28 \times 28 \times 32$ volume.
  * **Benefit:** This saves a massive amount of computational power in later layers.

- ##### ***B. Adding Non-Linearity***

  Even if you don't want to shrink the volume (e.g., keeping it at 192 channels), adding a $1 \times 1$ layer adds another **ReLU** activation. This allows the network to learn more complex, "non-trivial" functions without drastically increasing the parameter count.

<br>

#### 3. The "Network in Network" Perspective

The term comes from a 2013 paper by Lin et al. The idea is that instead of just having a simple linear filter, you are essentially embedding a small **Fully Connected (FC) neural network** that slides over every single pixel.

> **Expert Insight:** AI pioneer Yann LeCun notes that there is no fundamental difference between a "Fully Connected" layer and a convolution layer. A $1 \times 1$ convolution is essentially an FC layer applied at every spatial location.

<br>

**Key Paper:** [Lin et al., 2013. Network in network](https://arxiv.org/abs/1312.4400)

### Inception network motivation



- When you design a CNN you have to decide all the layers yourself. Will you pick a 3 x 3 Conv or 5 x 5 Conv or maybe a max pooling layer. You have so many choices.
- What **inception** tells us is, Why not use all of them at once?
<br>

- **The "Naive" Inception Module :**
  ![](https://github.com/doomguy0991/dls_c/blob/main/C4%20-%20Convolutional%20Neural%20Networks/Notes/Images/13.png?raw=1)
  Instead of picking one filter size, an Inception module performs $1 \times 1$, $3 \times 3$, $5 \times 5$ convolutions, and Max Pooling simultaneously. Their outputs are then concatenated (stacked) together.
  - **Goal:** Let the network learn and decide which features (small or large) are most important.
  - **Result:** A much more flexible layer, but it comes with a high "price tag" in terms of computation.
  - [[Szegedy et al. 2014. Going deeper with convolutions]](https://arxiv.org/abs/1409.4842)

<br>

#### The Computational Cost Problem

- Applying large filters (like $5 \times 5$) directly to a high-volume input is very "expensive" for a computer.

  **The Math (Without $1 \times 1$ Convs):**

  * **Input:** $28 \times 28 \times 192$
  * **Target Output:** $28 \times 28 \times 32$ (using 32 filters of size $5 \times 5$)
  * **Total Multiplications:**

  $$28 \times 28 \times 32 \times (5 \times 5 \times 192) \approx 120 \text{ Million operations}$$



  Performing 120 million multiplications for just one part of one layer is too slow for most practical applications.

<br>

#### The Solution: The "Bottleneck" Layer

- We can use a $1 \times 1$ convolution as a **"Bottleneck"** to shrink the number of channels before doing the heavy $5 \times 5$ math.

  **The Math (With $1 \times 1$ Bottleneck):**

  1. **Shrink:** Apply 16 filters of $1 \times 1$ to the $192$ channels.
  * Cost: $28 \times 28 \times 16 \times 1 \times 1 \times 192 \approx 2.4 \text{ Million}$


  2. **Process:** Apply the $5 \times 5$ filters to this smaller $16$-channel volume.
  * Cost: $28 \times 28 \times 32 \times 5 \times 5 \times 16 \approx 10 \text{ Million}$


  3. **Total Cost:** **$12.4 \text{ Million operations}$** (nearly **10x faster** than the original 120M).

<br>

#### The Final Inception Module (Reduced Dimension)

- By adding these $1 \times 1$ "bottlenecks" before every $3 \times 3$ and $5 \times 5$ convolution, the network stays powerful but runs significantly faster.
- Surprisingly, shrinking the data this way **does** **not** **hurt the model's accuracy.**

<br>

- **Inception module**, dimensions reduction version:
  - ![](https://github.com/doomguy0991/dls_c/blob/main/C4%20-%20Convolutional%20Neural%20Networks/Notes/Images/14.png?raw=1)
- Example of inception model in Keras:
  - ![](https://github.com/doomguy0991/dls_c/blob/main/C4%20-%20Convolutional%20Neural%20Networks/Notes/Images/inception_block1a.png?raw=1)


### GoogLeNet: The Inception Network



The **Inception Network** (also known as **GoogLeNet**) is built by stacking multiple Inception modules on top of each other. The name is a nod to the "Inception" movie meme: *"We need to go deeper."*

<br>

#### 1. Structure of the Network
A full Inception network isn't just one block; it is a long chain of Inception modules, occasionally separated by Max-Pooling layers to reduce the height and width of the data.

![](https://github.com/doomguy0991/dls_c/blob/main/C4%20-%20Convolutional%20Neural%20Networks/Notes/Images/15.png?raw=1)

*   **Input:** Usually starts with standard Convolution and Pooling layers (often called the "stem").
*   **Body:** A series of Inception modules stacked together.
*   **Final Output:** Ends with Global Average Pooling and a Softmax layer for classification.

<br>

#### **Intermediate Softmax Branches (Side Branches)**
One unique feature of the original GoogLeNet is the addition of **side branches** that come out of the middle of the network.



- ##### **What are they?**
  These are "mini-networks" that take the output of an intermediate layer, pass it through a few more layers, and try to make a prediction using their own Softmax output.

- ##### **Why use them?**
  1.  **Ensuring Feature Quality:** They ensure that the features being learned in the middle of the network are actually useful for identifying the final categories.
  2.  **Regularization:** They help prevent the model from **overfitting** by acting as a regularizer.
  3.  **Preventing Vanishing Gradients:** In very deep networks, the "signal" from the main loss at the end can get lost before it reaches the beginning layers. These side branches provide extra "gradient signals" to help the earlier layers learn more effectively.

  *Note: In later versions like Inception v2 and v3, researchers found these weren't always necessary, but they were a key innovation in the original version.*

<br>

#### **Key Takeaways & Evolution**
*   **Flexibility:** Instead of choosing a $3 \times 3$ or $5 \times 5$ filter, the network uses all of them and concatenates the results.
*   **Efficiency:** By using $1 \times 1$ "bottleneck" layers, the network stays deep and wide without exploding the computational cost.
*   **Versions:** Since the original paper, newer versions have been released:
    *   **Inception v2 & v3:** Improved performance and efficiency.
    *   **Inception v4:** Even deeper and more structured.
    *   **Inception-ResNet:** Combines the Inception module with ResNet’s "skip connections."

<br>

**Source Paper:** [Szegedy et al., 2014, Going Deeper with Convolutions](https://arxiv.org/abs/1409.4842)

### MobileNet: Efficiency via Depthwise Separable Convolutions



Standard neural networks (like ResNet or Inception) are powerful but computationally "expensive." If you want to run a model on a device with limited power—like a mobile phone—you need a more efficient architecture. This is where **MobileNet** and **Depthwise Separable Convolutions** come in.

<br>

#### The Problem: Normal Convolution Cost

In a normal convolution, every filter looks at **all** input channels at once.

  - Before breaking down the steps, remember the standard formula for any convolution or pooling layer to determine the output height ($H_{out}$) and width ($W_{out}$):

  $$H_{out} = \lfloor \frac{H_{in} + 2P - K}{S} + 1 \rfloor$$
  $$W_{out} = \lfloor \frac{W_{in} + 2P - K}{S} + 1 \rfloor$$

  *   **$H_{in} / W_{in}$**: Input Height/Width
  *   **$P$**: Padding
  *   **$K$**: Filter (Kernel) Size
  *   **$S$**: Stride
- * **Example Setup:**
  * **Input:** $6 \times 6 \times 3$ (3 channels: Red, Green, Blue)
  * **Filters:** $3 \times 3 \times 3$ (We want 5 of these)
  * **Output:** $4 \times 4 \times 5$



**Total Multiplications:**
To calculate the cost, we multiply:


$$\text{(Filter Size)} \times \text{(Filter Positions)} \times \text{(Number of Filters)}$$

$$(3 \times 3 \times 3) \times (4 \times 4) \times 5 = 2,160 \text{ operations}$$

<br>

#### The Solution: Depthwise Separable Convolution

Instead of doing everything in one heavy step, we break the process into **two smaller steps**.



- ##### **Step A: Depthwise Convolution**

  Instead of one filter looking at all channels, we use **one filter per channel**.

  - *   **Input Dimensions:** $(H_{in}, W_{in}, C_{in})$
    *   **Filter:** $K \times K$ (applied to each of the $C_{in}$ channels)
    *   **Intermediate Dimensions:** $(H_{int}, W_{int}, C_{in})$

  > **Key Rule:** The number of channels **does not change** in the depthwise step. If you start with 3 channels (RGB), you end with 3 channels. Only the Height and Width change based on the spatial formula above.
  * The Red filter only looks at the Red channel.
  * The Green filter only looks at the Green channel.
  * **Cost:** $(3 \times 3) \times (4 \times 4) \times 3 = 432 \text{ operations}$.

- ##### **Step B: Pointwise Convolution ($1 \times 1$)**

  *   **Input Dimensions:** $(H_{int}, W_{int}, C_{in})$
  *   **Filter:** $1 \times 1 \times C_{in}$ (applied $C_{out}$ times)
  *   **Final Output Dimensions:** $(H_{int}, W_{int}, C_{out})$

  > **Key Rule:** Because the filter is $1 \times 1$ and the stride is usually 1, the **Height and Width do not change** in this step. Only the number of channels ($C_{out}$) is transformed.

  Now we have an intermediate output ($4 \times 4 \times 3$), but we need $5$ channels for our final output. We use a $1 \times 1$ convolution (which you learned about in the "Network in Network" section) to map these 3 channels into 5.

  * **Cost:** $(1 \times 1 \times 3) \times (4 \times 4) \times 5 = 240 \text{ operations}$.

<br>

#### Comparison: How much did we save?

By splitting the work, the total cost for the **Depthwise Separable Convolution** is:


$$432 + 240 = 672 \text{ operations}$$

Compared to the original **2,160**, that is only **31%** of the cost!

| Feature | Normal Convolution | Depthwise Separable |
| --- | --- | --- |
| **Steps** | 1 Step | 2 Steps (Depthwise + Pointwise) |
| **Complexity** | High | Low (Mobile-friendly) |
| **Example Cost** | 2,160 operations | 672 operations |

<br>
<br>


#### The General Formula for Savings

The authors of the MobileNet paper showed that the ratio of cost between this new method and the old method is roughly:


$$\frac{1}{n_c'} + \frac{1}{f^2}$$


*(Where $n_c'$ is the number of output channels and $f$ is the filter size.)*

Since $f$ is usually 3 (making $1/f^2 = 1/9$), a MobileNet is often **10 times cheaper** to run than a standard CNN while achieving similar results.

> **Visual Note:** In complex diagrams, Depthwise Convolution is often represented by a stack of individual 2D filters, while Pointwise is shown as a thin $1 \times 1$ block.

Are you planning to implement these layers from scratch in a framework like TensorFlow, or are you focusing more on the mathematical optimization side for now?

### MobileNet v1 vs. v2: Key Differences



**MobileNet v1** revolutionized mobile AI by replacing expensive standard convolutions with **Depthwise Separable Convolutions**. It repeats this basic block 13 times before ending with Pooling, a Fully Connected layer, and Softmax.

**MobileNet v2** improves on this with two major innovations: **Residual Connections** and **Bottleneck Blocks**.

<br>

#### 1. The MobileNet v2 "Bottleneck" Block

Instead of just two steps, v2 uses a three-step process to learn more complex features while saving memory.

1. **Expansion ($1 \times 1$ Conv):** Increases the number of channels (usually by a factor of **6**). This allows the network to learn "richer" information.
2. **Depthwise Convolution:** Performs the spatial filtering on the expanded data.
3. **Projection ($1 \times 1$ Conv):** Shrinks the channels back down. This is crucial for **memory efficiency**, as it reduces the amount of data passed to the next layer.

<br>

#### 2. Residual (Skip) Connections

Borrowed from ResNet, MobileNet v2 adds **Residual Connections** that pass the input of a block directly to the output.

* **Why?** It helps gradients flow better during training, making it easier to train deeper networks without performance dropping.
* **Structure:** This block is repeated **17 times** in the v2 architecture.

<br>

### EfficientNet: Smart Scaling for Any Device



While MobileNet makes layers efficient, **EfficientNet** provides a way to scale an entire network up or down to fit the specific "computational budget" of your device (e.g., a high-end smartphone vs. a low-power edge device).

<br>

#### 1. The Three Dimensions of Scaling

To make a model more accurate (scaling up) or faster (scaling down), you can change three things:

1. **Depth ($d$):** Adding or removing layers.
2. **Width ($w$):** Increasing or decreasing the number of channels in layers.
3. **Resolution ($r$):** Using higher or lower-quality input images.

<br>

#### 2. Compound Scaling

The breakthrough of EfficientNet is **Compound Scaling**. Instead of just making a network super deep or super wide, the authors found that scaling all three ($r, d, w$) together at specific, balanced rates gives the best results.

* **The Problem:** If you only increase depth, the network eventually hits a point where adding more layers doesn't help much.
* **The Solution:** EfficientNet uses a formula to scale $r, d,$ and $w$ simultaneously. This ensures that if you give the model more "brain power" (computation), it uses it in the most effective way possible.

<br>

#### 3. Summary

* **Goal:** Get the highest accuracy for a specific amount of memory and speed.
* **Method:** Use **Compound Scaling** to balance Resolution, Depth, and Width.
* **Usage:** Instead of guessing how to tune your model, you can use open-source **EfficientNet** versions (B0 through B7) that are already optimized for different levels of computing power.

## Prctical Advice for Using Convnets

### Transfer Learning in Computer Vision



**Transfer Learning** is the process of taking a neural network already trained on a massive dataset (like ImageNet or MS COCO) and repurposing it for your specific task. In Computer Vision, this is almost always better than training from scratch.

<br>

#### 1. Why use it?

* **Speed:** Large models take weeks to train on hundreds of GPUs. You can download these "pre-trained weights" in seconds.
* **Small Data:** If you only have a few dozen photos of a specific object (e.g., your pet cats, **Tigger** and **Misty**), a model starting from random weights will overfit. A pre-trained model already knows how to detect edges, shapes, and textures.

<br>

#### 2. Implementation Strategies

Your strategy depends on how much data you have:

| Data Size | Action | Description |
| --- | --- | --- |
| **Very Small** | **Freeze All + Replace Softmax** | Keep all pre-trained layers "frozen" (weights don't change). Only train a new Softmax layer for your specific classes. |
| **Medium** | **Freeze Early + Train Late** | Freeze the first few layers (which detect general shapes) and train the later layers + Softmax. |
| **Large** | **Fine-Tuning** | Use pre-trained weights as a "starting point" instead of random numbers, then train the **entire** network on your data. |

<br>

#### 3. The "Save to Disk" Trick

If you are freezing most of the network, the output of those frozen layers for any given image will **never change**.

* **Optimization:** Instead of running the image through the frozen layers in every epoch, run them once, save those values (activations) to your hard drive, and train your new Softmax layer directly on those saved features. This makes training incredibly fast.

<br>

#### 4. Key Takeaway

Unless you have an exceptionally large dataset and a massive computing budget, you should **almost always** use transfer learning for computer vision applications.

Are you planning to use a specific pre-trained architecture like **ResNet** or **MobileNet** for your thesis project?


### Data Augmentation



In Computer Vision, we almost always feel like we don't have enough data. **Data Augmentation** is a technique to artificially increase the size of your training set by creating modified versions of your existing images. This helps the model generalize better and reduces overfitting.

<br>


#### Common Augmentation Methods

1. **Mirroring:** Flipping the image horizontally. For most tasks (like cat detection), a mirrored cat is still a cat. This is a very simple and effective way to double your data.
2. **Random Cropping:** Taking different "crops" or segments of the original image. As long as the crop still contains the main object, it counts as a unique new training example.
3. **Rotation & Shearing:** Rotating the image or distorting its shape. These are useful but are used slightly less often because they can be more complex to implement correctly.


<br>


#### Color Shifting

Color shifting involves distorting the Red, Green, and Blue (RGB) channels of an image.

* **The Goal:** It makes the model robust to different lighting conditions. For example, a cat in yellow sunlight should be recognized just as easily as a cat in blue-tinted indoor light.
* **PCA Augmentation:** A more advanced method (used in the famous AlexNet paper) that uses Principal Component Analysis to change colors in a way that keeps the overall "tint" realistic while still providing variety.



<br>

#### Implementing Distortions (Parallelism)

When working with huge datasets, you don't want to manually save all distorted images to your hard drive. Instead, we implement it **"on the fly"**:

* **CPU Threads:** One part of your computer (the CPU) is responsible for loading the raw images and applying the random distortions (flipping, cropping, color shifting).
* **GPU Training:** While the CPU is preparing the next batch of distorted images, the GPU is busy training the model.
* **Efficiency:** This parallel process means the training never has to wait for the images to be ready.


<br>


#### Summary for Practice

Data augmentation has its own **hyperparameters** (like how much to crop or how much to shift colors). A good starting point is usually to look at open-source implementations of famous models and see which settings worked best for them.

### The State of Computer Vision



In the world of AI, different fields (like speech recognition or natural language processing) have different amounts of data. Computer Vision is unique because, even though we have millions of images, the problem is so complex that it often feels like we are still in a "small data" regime.


<br>


#### Data vs. Hand-Engineering

There is a spectrum in machine learning: the more data you have, the less "hand-engineering" you need.

* **Lots of Data:** You can use simpler algorithms and giant neural networks. The model learns everything from the data itself.
* **Little Data:** You need more hand-engineering. This means carefully designing the network architecture (like ResNet or Inception) and using "hacks" or insights to make the model perform well.
* **Object Detection:** This field has even less data than image classification because labeling bounding boxes is expensive and slow.


<br>


#### Benchmarks vs. Production

If you read research papers, you will see techniques used to win competitions or top the "benchmarks." However, these are often not practical for real-world products.

* **Ensembling:** Training 3 to 15 different models independently and averaging their outputs ($y$-hats). It might give you a 1-2% boost in accuracy, but it makes your system 3-15 times slower and uses much more memory.
* **Multi-crop at Test Time:** Running the same test image through the model 10 times (taking the center, the four corners, and their mirrored versions) and averaging the results. Like ensembling, this is great for winning a contest but usually too slow for a real-time application.


<br>


#### Practical Advice for Practitioners

Because Computer Vision relies so heavily on complex architectures, you don't always need to "reinvent the wheel."

1. **Use Open Source:** Start with an architecture that someone else has already designed and tested.
2. **Transfer Learning:** Download weights that were trained on huge datasets (like ImageNet). Someone else has already spent weeks and thousands of dollars on GPUs to find those weights—you can just use them as a starting point.
3. **Fine-Tuning:** Use those pre-trained weights and adjust them slightly for your specific task. This is almost always faster and more accurate than starting from zero.

## Object detection

> Learn how to apply your knowledge of CNNs to one of the toughest but hottest field of computer vision: Object detection.


### Object Localization






- Object detection is one of the areas in which deep learning is doing great in the past two years.

- What are localization and detection?

  - **Image Classification**:
    - Classify an image to a specific class. The whole image represents one class. We don't want to know exactly where are the object. Usually only one object is presented.
    - ![](https://github.com/doomguy0991/dls_c/blob/main/C4%20-%20Convolutional%20Neural%20Networks/Notes/Images/Classification.jpg?raw=1)
  - **Classification with localization**:
    - Given an image we want to learn the class of the image and where are the class location in the image. We need to detect a class and a rectangle of where that object is. Usually only one object is presented.
    - ![](https://github.com/doomguy0991/dls_c/blob/main/C4%20-%20Convolutional%20Neural%20Networks/Notes/Images/ClassificationLoc.jpg?raw=1)
  - **Object detection**:
    - Given an image we want to detect all the object in the image that belong to a specific classes and give their location. An image can contain more than one object with different classes.
    - ![](https://github.com/doomguy0991/dls_c/blob/main/C4%20-%20Convolutional%20Neural%20Networks/Notes/Images/ObjectDetection.png?raw=1)
  - **Semantic Segmentation**:
    - We want to Label each pixel in the image with a category label. Semantic Segmentation Don't differentiate instances, only care about pixels. It detects no objects just pixels.
    - If there are two objects of the same class is intersected, we won't be able to separate them.
    - ![](https://github.com/doomguy0991/dls_c/blob/main/C4%20-%20Convolutional%20Neural%20Networks/Notes/Images/SemanticSegmentation.png?raw=1)
  - **Instance Segmentation**
    - This is like the full problem. Rather than we want to predict the bounding box, we want to know which pixel label but also distinguish them.
    - ![](https://github.com/doomguy0991/dls_c/blob/main/C4%20-%20Convolutional%20Neural%20Networks/Notes/Images/InstanceSegmentation.png?raw=1)

- To make image classification we use a Conv Net with a Softmax attached to the end of it.

- To make classification with localization we use a Conv Net with a softmax attached to the end of it and a four numbers `bx`, `by`, `bh`, and `bw` to tell you the location of the class in the image. The dataset should contain this four numbers with the class too.

- Defining the target label Y in classification with localization problem:

  - ```
    Y = [
      		Pc				# Probability of an object is presented
      		bx				# Bounding box
      		by				# Bounding box
      		bh				# Bounding box
      		bw				# Bounding box
      		c1				# The classes
      		c2
      		...
    ]
    ```

  - Example (Object is present):

    - ```
      Y = [
        		1		# Object is present
        		0
        		0
        		100
        		100
        		0
        		1
        		0
      ]
      ```

  - Example (When object isn't presented):

    - ```
      Y = [
        		0		# Object isn't presented
        		?		# ? means we dont care with other values
        		?
        		?
        		?
        		?
        		?
        		?
      ]
      ```

<br>


#### Mapping Components to Loss Functions

When training an object localization model, the vector $Y$ is a hybrid of different data types. Because we are trying to predict a probability, a category, and a coordinate all at once, we use a **multi-task loss** function.

Here is the breakdown of which loss applies to which part:




| Component | Task Type | Best Loss Function |
| --- | --- | --- |
| **$P_c$** (Object Presence) | Binary Classification | **Logistic Regression Loss** (Binary Cross-Entropy) |
| **$b_x, b_y, b_h, b_w$** (Box) | Regression | **Squared Error** (Mean Squared Error / $L_2$ loss) |
| **$c_1, c_2, c_3$** (Class Labels) | Multi-class Classification | **Softmax Loss** (Categorical Cross-Entropy) |

#### Why do we split them?

1. **Logistic Regression Loss for $P_c$:**
Since $P_c$ is a probability (is there an object or not?), we treat it as a binary classification. Logistic regression loss is mathematically optimized to push predictions toward exactly $0$ or $1$.




2. **Squared Error for Bounding Box:**
The coordinates ($b_x, b_y$) and dimensions ($b_h, b_w$) are continuous real numbers. We use squared error (or sometimes $L_1$ loss) to minimize the physical distance between the predicted box and the ground truth box.




3. **Softmax Loss for Classes:**
Because the object can only belong to one category (Car **or** Pedestrian **or** Motorcycle), we use Softmax to create a probability distribution across all classes that sums to $1$. The Softmax loss then penalizes the model based on how far the predicted class probability is from the true "1" in the one-hot encoded vector.

### The Combined Loss Formula

In your code, the total loss ($L$) for a single training example where an object is present ($P_c = 1$) would look something like this:

$$L(\hat{y}, y) = \underbrace{L_{logistic}(\hat{P_c}, P_c)}_{\text{Presence}} + \underbrace{\sum_{i=2}^5 (\hat{y}_i - y_i)^2}_{\text{Bounding Box}} + \underbrace{L_{softmax}(\hat{c}, c)}_{\text{Classification}}$$

### Landmark Detection



Landmark detection is a more general version of object localization. Instead of just drawing a bounding box, the neural network is trained to output the specific $X$ and $Y$ coordinates of important points—called **landmarks**—within an image.

<br>

#### How it Works (The Output Vector)

The neural network ends with a set of output units that predict the coordinates for $N$ specific points. For example, if you want to track 64 points on a face to detect expressions or apply AR filters:

* **$P_c$:** Is there a face? (1 output)
* **Landmarks:** $(l_{1x}, l_{1y}), (l_{2x}, l_{2y}) \dots (l_{64x}, l_{64y})$ (128 outputs)
* **Total Output Size:** 129 units.

<br>


#### Key Applications

1. **Face Recognition & Emotion Detection:** By tracking the corners of the eyes, the shape of the mouth, and the jawline, the network can tell if someone is smiling, frowning, or blinking.
2. **Augmented Reality (AR):** This is the building block for apps like Snapchat. By knowing exactly where the "top of the head" or "nose bridge" is, the software can accurately place a digital crown or glasses on the user.
3. **Pose Estimation:** By defining landmarks for joints (shoulders, elbows, wrists, knees), the network can recognize the "pose" or movement of a person, which is used in sports analysis or gesture control.

<br>


#### The Consistency Rule

For landmark detection to work, the **labels must be consistent** across the entire training dataset. For example, if $l_1$ is defined as the "outer corner of the left eye," it must represent that exact spot in every single image you label.

While annotating thousands of images with 60+ points is laborious, it allows the network to learn highly detailed "non-trivial" spatial functions.

### Object Detection: Sliding Windows Algorithm



To detect objects like cars in a large image, we use the **Sliding Windows** approach. This technique turns a **simple image classifier into a full detection system.**


<br>

#### The Training Phase

Before you can detect objects, you must train a ConvNet to recognize them using "closely cropped" images.
- ![](https://github.com/doomguy0991/dls_c/blob/main/C4%20-%20Convolutional%20Neural%20Networks/Notes/Images/18.png?raw=1)
* **Dataset:** Collect images where the car is centered and fills the frame ($y=1$) and images with no cars ($y=0$).
* **Goal:** The network learns exactly what a car looks like when it is perfectly framed.


<br>



#### The Detection Phase

Once trained, you use the model on a large test image by "sliding" a search window over it:

1. **Select a Window Size:** Start with a small square.
2. **Slide with Stride:** Move the square across the image. At each position, crop the pixels inside and feed them into the ConvNet.
3. **Multiple Sizes:** Repeat the process using larger windows to find objects that are closer to the camera.

4. Store the rectangles that contains the cars.

5. If two or more rectangles intersects choose the rectangle with the best accuracy.

<br>


#### The Major Disadvantage: Computational Cost

The main problem with this method is **speed**.

* **High Workload:** You have to run the ConvNet hundreds or thousands of times just to process one image.
* **Infeasibility:** While this worked for old, simple linear classifiers, modern Deep Learning models are too complex to be run this many times independently. It is simply too slow for real-time use.


<br>


#### Moving Toward Efficiency

To fix the speed issue, researchers developed a way to perform this entire process **convolutionally**. Instead of cropping the image into pieces, we pass the entire image through the network once to get all the predictions at the same time.

### Convolutional Implementation of Sliding Windows



- Turning FC layer into convolutional layers (predict image class from four classes):
  - ![](https://github.com/doomguy0991/dls_c/blob/main/C4%20-%20Convolutional%20Neural%20Networks/Notes/Images/19.png?raw=1)
  - As you can see in the above image, we turned the FC layer into a Conv layer using a convolution with the width and height of the filter is the same as the width and height of the input.
- **Convolution implementation of sliding windows**:
  - First lets consider that the Conv net you trained is like this (No FC all is conv layers):
    - ![](https://github.com/doomguy0991/dls_c/blob/main/C4%20-%20Convolutional%20Neural%20Networks/Notes/Images/20.png?raw=1)
  - Say now we have a 16 x 16 x 3 image that we need to apply the sliding windows in. By the normal implementation that have been mentioned in the section before this, we would run this Conv net four times each rectangle size will be 16 x 16.
  - The convolution implementation will be as follows:
    - ![](https://github.com/doomguy0991/dls_c/blob/main/C4%20-%20Convolutional%20Neural%20Networks/Notes/Images/21.png?raw=1)
  - Simply we have feed the image into the same Conv net we have trained.
  - The left cell of the result "The blue one" will represent the the first sliding window of the normal implementation. The other cells will represent the others.
  - Its more efficient because it now shares the computations of the four times needed.
  - Another example would be:
    - ![](https://github.com/doomguy0991/dls_c/blob/main/C4%20-%20Convolutional%20Neural%20Networks/Notes/Images/22.png?raw=1)
  - This example has a total of 16 sliding windows that shares the computation together.
  - [[Sermanet et al., 2014, OverFeat: Integrated recognition, localization and detection using convolutional networks]](https://arxiv.org/abs/1312.6229)

#### Why This is a Game Changer
- **Speed**: It allows the network to "share" computations in overlapping regions. You only calculate the features for the middle pixels once, even though they belong to all four sliding windows.

- **Efficiency**: One single forward pass can provide predictions for hundreds of windows simultaneously.


#### The Remaining Weakness
While this method is extremely fast, it still has a precision problem:
- the bounding boxes are limited by the stride of the network (often 2 or 4 pixels). If a car is "between" two windows, the box won't perfectly frame it.


Next, we'll look at the YOLO (You Only Look Once) algorithm, which uses this convolutional foundation but adds a way to predict much more accurate bounding boxes.
- The weakness of the algorithm is that the position of the rectangle wont be so accurate. Maybe none of the rectangles is exactly on the object you want to recognize.
  - ![](https://github.com/doomguy0991/dls_c/blob/main/C4%20-%20Convolutional%20Neural%20Networks/Notes/Images/23.png?raw=1)
  - In red, the rectangle we want and in blue is the required car rectangle.